<div id="singlestore-header" style="display: flex; background-color: #525252; padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/brand/chart-network.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%; color: #FFFFFF;">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px; color: #FFFFFF;">How to Build LLM Apps that can See Hear Speak</h1>
    </div>
</div>

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This tutorial is meant for Standard &amp; Premium Clusters. You can't run this with a Free Starter Cluster due to restrictions on Storage. Create a Cluster using +group in the left nav &amp; select Standard for this notebook. Gallery notebooks tagged with "Starter" are suitable to run on a Free Starter Cluster </p>
    </div>
</div>

<a name="architecture"></a>
<a name="contents"></a>

# What you'll build

![SingleStore LLM App](https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/notebooks/how-to-build-llm-apps-that-can-see-hear-speak/images/singlestore-llm-app.png)

By the end of this notebook you will have a small application that answers questions about company
financials in plain English, and that can see, hear and speak. Three years of quarterly results for
Amazon, Tesla and Palantir are loaded from files that ship with this notebook; a LangChain SQL agent
turns your questions into MySQL and answers in prose;
each question is cached alongside its embedding, so a later question that *means* the same thing is
answered from the cache; the answer is spoken and transcribed back; and a chart drawn from the stored rows is
described by a vision model.

All of it lives in **one** SingleStore database. The rows, the embeddings and the cache sit together, so
the cache lookup is an ordinary SQL query over a `VECTOR` column — there is no second system to keep in
sync.

- [Step 1](#ddl) — one database and three tables, including the `VECTOR` columns the cache needs
- [Step 2](#imports) — install LangChain, enter your OpenAI key, open a connection
- [Step 3](#ingest) — load quarterly financials for three companies, taken from their SEC filings
- [Step 4](#connect_s2) — ask questions in English, then put a semantic cache in front of the agent
- [Step 5](#speech) — speak the answer, play it, transcribe it back
- [Step 6](#image) — chart the data and have a vision model describe the chart

Run the cells in order. Every cell is safe to run again.
[Troubleshooting](#troubleshooting) at the end lists the errors worth knowing about in advance.

## Before you start

You need a Standard or Premium cluster, as the alert above says, with a database selected in the
drop-down at the top of this notebook — every SQL cell runs against whatever is chosen there.

You need one account: **OpenAI**, for the models. A full run costs a few cents. The data needs no
account at all — it ships with the notebook (see step 3).

Budget about five minutes, most of it the agent thinking in step 4.

<a name="ddl"></a>
- [Back to Contents](#contents)
## Step 1: Create the tables

First a database named `llm_webinar`. Expect a one-line success message and nothing else.

Running these cells twice is harmless. The database and the `embeddings` cache use `IF NOT EXISTS`, so
the cache step 4 builds survives. The two data tables, `companyInfo` and `financials`, are dropped and recreated
instead, because step 3 reloads them from data held in the notebook at no cost. The same cell also drops
`stockTable`, `newsSentiment` and `earningsCalls`, tables left by earlier versions of this notebook.
Every statement names `llm_webinar` explicitly, so nothing in another database is touched. To start over deliberately, the [Reset Demo](#reset)
cell at the end drops the database outright.

In [1]:
%%sql
CREATE DATABASE IF NOT EXISTS llm_webinar;

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p> Make sure to select a database from the drop-down menu at the top of this notebook. It updates the <tt>connection_url</tt>  to connect to that database.</p>
    </div>
</div>

### The three tables

Two tables hold what step 3 loads: **`companyInfo`**, one row per company — name, industry, exchange,
fiscal year end — and **`financials`**, one row per company per quarter with revenue, operating income
and net income in US dollars. `companyInfo` is a `REFERENCE` table because it is tiny — SingleStore
replicates it to every node, so joins against it never move data across the cluster.

The third, **`embeddings`**, is the semantic cache: a question, its answer, and an embedding of
each. Those embedding columns are `VECTOR(1536, F32)`, SingleStore's native vector type, sized for
what `text-embedding-3-small` produces. Using the real type rather than a blob means the dimension
of every value is validated on insert, results print readably, and a vector index can be added
later — which is what `NOT NULL` is there for.

The money columns are `BIGINT` rather than `FLOAT`, because a quarter of Amazon revenue is about
$200 billion, more digits than a `FLOAT` holds exactly.

The `SHOW TABLES` cell that follows is your check: expect exactly these three names. If it comes back
empty, the database drop-down at the top of the notebook is pointing somewhere else.

In [2]:
%%sql
-- Every name is qualified with llm_webinar, so this cell can never drop or create tables in
-- whichever database happens to be selected in the drop-down.

-- Tables from earlier versions of this notebook, which loaded data from a stock-data API.
DROP TABLE IF EXISTS llm_webinar.stockTable;
DROP TABLE IF EXISTS llm_webinar.newsSentiment;
DROP TABLE IF EXISTS llm_webinar.earningsCalls;

-- The two data tables are rebuilt on every run: step 3 reloads them from the notebook at no
-- cost, and rebuilding means an older copy of either can never be left with stale columns.
DROP TABLE IF EXISTS llm_webinar.companyInfo;
CREATE ROWSTORE REFERENCE TABLE llm_webinar.companyInfo (
    ticker VARCHAR(10) PRIMARY KEY,
    name VARCHAR(200),
    cik CHAR(10),                       -- the company's SEC identifier
    industry VARCHAR(200),              -- SEC's SIC description
    exchange VARCHAR(20),
    fiscal_year_end CHAR(5),            -- MM-DD
    state_of_incorporation CHAR(2)
);

DROP TABLE IF EXISTS llm_webinar.financials;
CREATE TABLE llm_webinar.financials (
    ticker VARCHAR(10),
    fiscal_year INT,
    fiscal_quarter TINYINT,             -- 1 to 4
    period_end DATE,
    q4_derived BOOL,                    -- 1 when Q4 is the full year minus Q1 to Q3
    revenue BIGINT,                     -- US dollars
    operating_income BIGINT,
    net_income BIGINT,
    SORT KEY (ticker, period_end),
    SHARD KEY (ticker)
);

CREATE TABLE IF NOT EXISTS `llm_webinar`.`embeddings` (
    `id` bigint(11) NOT NULL AUTO_INCREMENT,
    `category` varchar(255) CHARACTER SET utf8 COLLATE utf8_general_ci DEFAULT NULL,
    `question` longtext CHARACTER SET utf8 COLLATE utf8_general_ci,
    `question_embedding` VECTOR(1536, F32) NOT NULL,
    `answer` longtext CHARACTER SET utf8 COLLATE utf8_general_ci,
    `answer_embedding` VECTOR(1536, F32) NOT NULL,
    `created_at` datetime DEFAULT NULL,
    UNIQUE KEY `PRIMARY` (`id`) USING HASH,
    SHARD KEY `__SHARDKEY` (`id`),
    KEY `category` (`category`) USING HASH,
    SORT KEY `__UNORDERED` (`created_at` DESC)
);

In [3]:
%%sql
SHOW TABLES;

<a name="imports"></a>
- [Back to Contents](#contents)
## Step 2: Packages, keys and connections

The hosted image already ships `openai`, `numpy`, `pandas`, `requests` and `singlestoredb`, so the
installs are LangChain, for the agent in step 4, and matplotlib, for the chart in step 6. numpy is
pinned to the version already installed, so pip can't replace it under the running kernel. Expect
about thirty seconds of quiet output.

Two warnings you can ignore: `langchain-community` — the only home for the SQL toolkit step 4 uses
— announces on import that it is being sunset, and pip may mention conflicts among packages this
notebook never imports.

In [4]:
%pip install --quiet "langchain>=1.3,<2" "langchain-community>=0.4.2,<0.5" "langchain-openai>=1.5,<2"

# The hosted image has no matplotlib, which the chart in step 6 needs. numpy is pinned to the
# version already installed, so pip adds matplotlib without swapping numpy under the kernel.
# python-dateutil is capped because 4.x breaks the image's ibis-framework.
from importlib.metadata import version

%pip install --quiet "matplotlib<3.11" "numpy=={version('numpy')}" "python-dateutil<3"

Now the imports: standard library, the two SDKs, and the LangChain pieces step 4 needs. Nothing
here touches the network, so this cell returns instantly. A `NameError` further down usually means
this cell has not run in the current kernel.

In [5]:
import csv
import getpass
import io
import json
import os
import time
from datetime import datetime, timedelta

import numpy as np
import openai
import requests
import singlestoredb as s2

# LangChain moved these in 1.x: `langchain.sql_database` and
# `langchain.agents.agent_toolkits` no longer exist. These are the current paths.
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_openai import ChatOpenAI

import langchain

print(f'openai {openai.__version__} | langchain {langchain.__version__}')

# Report whether this image's brotli is one of the affected ones. This is informational
# only — the OpenAI client cell below sidesteps the decoder either way.
try:
    import brotli

    brotli.Decompressor().process(brotli.compress(b'probe'), output_buffer_limit=1 << 20)
    print(f'brotli {brotli.__version__}: accepts output_buffer_limit')
except TypeError:
    print(f'brotli {brotli.__version__}: too old for httpx2 (expected on this image) — '
          f'the OpenAI client below will not request brotli-encoded responses')
except ImportError:
    print('brotli not installed — nothing to work around')

### Set your OpenAI key

Run the next cell and paste your [OpenAI API key](https://platform.openai.com/api-keys) at the prompt.
OpenAI does everything model-shaped here: embeddings, the text-to-SQL agent, speech, transcription and
image description. `getpass` keeps the key off the screen and out of the saved notebook, which also
means re-entering it after a kernel restart.

In [6]:
openai_apikey = getpass.getpass('enter openai apikey here')

assert openai_apikey, 'An OpenAI key is required for embeddings, the SQL agent, speech and captioning.'

os.environ['OPENAI_API_KEY'] = openai_apikey

### Build the OpenAI client

One client, used by everything: embeddings, transcription, speech, captioning, and the LangChain
agent in step 4. The models for those jobs are chosen in the next cell. It ends with a deliberately cheap embeddings call, so a wrong or unfunded key fails
here rather than a dozen cells later. Expect two printed lines, the second reading
`text-embedding-3-small returned 1536 dimensions` — the size the `embeddings` table was declared
for. If you swap in a different embedding model, read the note on the similarity threshold in step 4
first: **the threshold is a property of the model**, and carrying one over is a silent failure.

The `http_client` argument turns off brotli compression, which the version on this image cannot
decode. Without it you get `APIConnectionError: Connection error` printed under a successful
`HTTP/1.1 200 OK` — an error that looks like the network or your key and is neither.

In [7]:
import httpx2
from openai import OpenAI

# Do not advertise brotli, so the image's brotli 1.1.0 is never asked to decode a response.
http_client = httpx2.Client(headers={'Accept-Encoding': 'gzip, deflate'})

client = OpenAI(api_key=openai_apikey, http_client=http_client)

print(f"Accept-Encoding: {client._client.headers.get('accept-encoding')}")

EMBEDDING_MODEL = 'text-embedding-3-small'   # 1536 dimensions, matches the DDL above


def get_embeddings(inputs: list[str], model: str = EMBEDDING_MODEL) -> list[list[float]]:
    """Return one embedding per input string."""
    return [x.embedding for x in client.embeddings.create(input=inputs, model=model).data]


# Fail here, on one cheap call, rather than midway through the ingest loops.
_probe = get_embeddings(['connectivity probe'])[0]
print(f'{EMBEDDING_MODEL} returned {len(_probe)} dimensions')

assert len(_probe) == 1536, (
    f'{EMBEDDING_MODEL} returned {len(_probe)} dimensions but the embeddings table declares '
    f'VECTOR(1536, F32). Change the DDL to match, or pick a 1536-dimension model.'
)

### Choose the models

OpenAI retires older models regularly, usually with at least six months' notice, and its
[List models](https://developers.openai.com/api/reference/resources/models/methods/list) endpoint
reports each model's planned shutdown date. Rather than trust model names written into this
notebook, this cell asks OpenAI which models your key can use right now and skips any that are
scheduled to shut down. It picks three: a chat model for the SQL agent and the captions, a speech
model, and a transcription model.

For each one it takes the first available entry from its `PREFERRED_..._MODELS` list. If every
entry has been retired, it falls back to the newest model of the same kind on your account,
preferring the low-cost tiers. Expect four lines naming the models it chose. A retirement then
shows up as a different name in this output rather than an error further down.

In [8]:
import openai

# OpenAI retires models on a schedule. This notebook used to hard-code gpt-4o-mini,
# gpt-4o-mini-tts and whisper-1, and all three are being shut down. So instead of pinning
# names, list the models we'd like for each job, in order of preference. This cell asks
# OpenAI which models your key can use right now and skips any with a shutdown date.
PREFERRED_CHAT_MODELS = ['gpt-6-luna', 'gpt-6.1-terra']   # SQL agent (step 4), captions (step 6)
PREFERRED_SPEECH_MODELS = ['gpt-audio-1.5']                # text to speech (step 5)
PREFERRED_TRANSCRIBE_MODELS = ['gpt-transcribe']           # speech to text (step 5)

LOW_COST = ('luna', 'nano', 'mini')
NOT_CHAT = ('audio', 'realtime', 'transcribe', 'tts', 'image', 'search', 'codex', 'cyber',
            'pro', 'chat', 'live', 'oss', 'embedding', 'moderation', 'preview', 'instruct',
            'translate', 'whisper')


def has_shutdown_date(model) -> bool:
    """True if OpenAI has announced a shutdown date for this model."""
    return bool(getattr(model, 'shutdown_date', None))


active_models = {m.id: m for m in client.models.list() if not has_shutdown_date(m)}


def pick_model(preferred: list[str], same_kind) -> str | None:
    """The first preferred model that is active, else the newest active one of the same kind."""
    for name in preferred:
        if name in active_models:
            return name

    # Nothing on the list is still active: take the newest model of the same kind,
    # low-cost tiers first.
    fallback = [m for m in active_models.values() if same_kind(m.id)]
    fallback.sort(key=lambda m: (any(word in m.id for word in LOW_COST), m.created), reverse=True)
    return fallback[0].id if fallback else None


CHAT_MODEL = pick_model(
    PREFERRED_CHAT_MODELS,
    lambda name: name.startswith('gpt-') and not any(word in name for word in NOT_CHAT))
SPEECH_MODEL = pick_model(
    PREFERRED_SPEECH_MODELS,
    lambda name: name.startswith('gpt-audio') and 'preview' not in name)
TRANSCRIBE_MODEL = pick_model(
    PREFERRED_TRANSCRIBE_MODELS,
    lambda name: 'transcribe' in name and not any(w in name for w in ('live', 'realtime', 'diarize')))

assert CHAT_MODEL, ('No active GPT chat model is available to this API key. '
                    'Check https://developers.openai.com/api/docs/models')

# Current models accept "no hidden reasoning step" and the most repeatable output, which keeps
# the agent quick and its answers repeatable. If a future model rejects
# either setting, CHAT_SETTINGS is left empty and the model's defaults are used instead.
CHAT_SETTINGS = {'reasoning_effort': 'none', 'temperature': 0}
try:
    client.chat.completions.create(
        model=CHAT_MODEL, messages=[{'role': 'user', 'content': 'say ok'}], **CHAT_SETTINGS)
except openai.BadRequestError as err:
    # Fall back only if the model objected to the settings; raise any other 400.
    complaint = f"{getattr(err, 'param', '') or ''} {err}"
    if 'reasoning_effort' not in complaint and 'temperature' not in complaint:
        raise
    CHAT_SETTINGS = {}

print(f'chat model:          {CHAT_MODEL} {CHAT_SETTINGS or "(model defaults)"}')
print(f'speech model:        {SPEECH_MODEL or "none available, step 5 cannot speak"}')
print(f'transcription model: {TRANSCRIBE_MODEL or "none available, step 5 cannot transcribe"}')

if EMBEDDING_MODEL in active_models:
    print(f'embedding model:     {EMBEDDING_MODEL} (no shutdown date announced)')
else:
    print(f'Note: {EMBEDDING_MODEL} has a shutdown date announced, or is not available to this '
          f'key. Pick a 1536-dimension replacement and re-check the similarity threshold in step 4.')

### Connect to SingleStore

Expect one line: `connected to llm_webinar; all three tables present`. If it reports a missing table
or no database selected, the drop-down at the top of the notebook is pointing somewhere other than
where step 1 created things.

The password in a notebook connection URL is a token that expires after about an hour, so the helpers
read the URL when they need it and let the driver refresh it. One connection then survives both a
token refresh and a change to the database drop-down.

In [9]:
def current_connection_url() -> str:
    """Return the connection URL as it is now, not as it was at kernel startup.

    The password is a JWT that expires after roughly an hour, so a copy taken earlier in the
    session may already be dead. The first two sources are read live.
    """
    url = None

    try:
        from singlestoredb.notebook import portal
        url = portal.connection_url
    except Exception:
        pass   # not running inside a SingleStore hosted notebook

    url = url or os.environ.get('SINGLESTOREDB_URL') or connection_url

    # SINGLESTOREDB_URL carries no scheme, and SQLAlchemy needs one to choose a dialect.
    return url if '://' in url else f'singlestoredb://{url}'


EXPECTED_TABLES = {'companyInfo', 'financials', 'embeddings'}


def s2_connect():
    """Open a connection that re-reads its credentials rather than pinning them."""
    # track_env=True discards any URL passed to connect() and reads SINGLESTOREDB_URL on every
    # statement, so only ask for it when that variable is actually set.
    if os.environ.get('SINGLESTOREDB_URL'):
        conn = s2.connect(track_env=True)
    else:
        conn = s2.connect(current_connection_url())

    try:
        with conn.cursor() as cur:
            # track_env defers connecting, so this statement is what opens the socket.
            cur.execute('SELECT DATABASE()')
            database = cur.fetchone()[0]

            cur.execute(
                'SELECT table_name FROM information_schema.tables '
                'WHERE table_schema = DATABASE()'
            )
            tables = {row[0] for row in cur.fetchall()}

    except s2.Error as exc:
        if exc.errno == 2628 or 'JWT' in str(exc):
            raise RuntimeError(
                'SingleStore rejected the notebook credentials: the JWT in the connection URL '
                'has expired. It is issued when the kernel starts and lasts about an hour, so a '
                'long session runs past it. Restart the kernel and run the cells again — the '
                'portal issues a fresh token at startup.'
            ) from exc
        raise

    assert database, (
        'no database is selected. Pick llm_webinar from the drop-down at the top of the '
        'notebook, then re-run this cell.'
    )

    assert database == 'llm_webinar', (
        f'the drop-down points at {database}, not llm_webinar. Select llm_webinar at the top of '
        f'the notebook, then re-run this cell. Step 3 and the cache write to llm_webinar by name, '
        f'and the agent reads whichever database is selected, so the two must match.'
    )

    missing = EXPECTED_TABLES - tables

    assert not missing, (
        f'connected to {database}, which is missing {sorted(missing)}. Either the DDL cells have '
        f'not been run, or the drop-down points at a different database.'
    )

    print(f'connected to {database}; all three tables present')

    return conn


s2_conn = s2_connect()

<a name="ingest"></a>
- [Back to Contents](#contents)
## Step 3: Load the data

The data is part of the notebook itself — the next cell holds it as two short blocks of CSV — so
loading it needs no account, no API key and no download. Nothing here can be rate-limited, paywalled
or blocked by a firewall.

The figures come from each company's filings with the US Securities and Exchange Commission, through
[SEC EDGAR](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)'s free XBRL
data: quarterly revenue, operating income and net income (attributable to common stockholders, the
`NetIncomeLoss` figure in each 10-K and 10-Q) in US dollars, from Q1 2023 to Q3 2026, for
Amazon, Tesla and Palantir — all three on calendar fiscal years. Companies report Q4 only inside the
full year, so each Q4 row is the full year minus the first three quarters, and is marked
`q4_derived = 1`.

Run the data cell, then the load cell. Expect two lines from the second, `companyInfo: 3 rows` and
`financials: 45 rows`. It empties both tables before loading, so running it again leaves the same rows
rather than duplicates.

In [10]:
# Snapshot of SEC EDGAR XBRL data, taken 2026-11-17. Amounts are US dollars.
COMPANIES_CSV = '''
ticker,name,cik,industry,exchange,fiscal_year_end,state_of_incorporation
AMZN,AMAZON COM INC,0001018724,Retail-Catalog & Mail-Order Houses,Nasdaq,12-31,DE
TSLA,"Tesla, Inc.",0001318605,Motor Vehicles & Passenger Car Bodies,Nasdaq,12-31,TX
PLTR,Palantir Technologies Inc.,0001321655,Services-Prepackaged Software,Nasdaq,12-31,
'''

FINANCIALS_CSV = '''
ticker,fiscal_year,fiscal_quarter,period_end,q4_derived,revenue,operating_income,net_income
AMZN,2023,1,2023-03-31,0,127358000000,4774000000,3172000000
AMZN,2023,2,2023-06-30,0,134383000000,7681000000,6750000000
AMZN,2023,3,2023-09-30,0,143083000000,11188000000,9879000000
AMZN,2023,4,2023-12-31,1,169961000000,13209000000,10624000000
AMZN,2024,1,2024-03-31,0,143313000000,15307000000,10431000000
AMZN,2024,2,2024-06-30,0,147977000000,14672000000,13485000000
AMZN,2024,3,2024-09-30,0,158877000000,17411000000,15328000000
AMZN,2024,4,2024-12-31,1,187792000000,21203000000,20004000000
AMZN,2025,1,2025-03-31,0,155667000000,18405000000,17127000000
AMZN,2025,2,2025-06-30,0,167702000000,19171000000,18164000000
AMZN,2025,3,2025-09-30,0,180169000000,17422000000,21187000000
AMZN,2025,4,2025-12-31,1,213386000000,24977000000,21192000000
AMZN,2026,1,2026-03-31,0,181519000000,23852000000,30255000000
AMZN,2026,2,2026-06-30,0,200606000000,27461000000,62647000000
AMZN,2026,3,2026-09-30,0,201944000000,28076000000,20685000000
TSLA,2023,1,2023-03-31,0,23329000000,2664000000,2513000000
TSLA,2023,2,2023-06-30,0,24927000000,2399000000,2703000000
TSLA,2023,3,2023-09-30,0,23350000000,1764000000,1853000000
TSLA,2023,4,2023-12-31,1,25167000000,2064000000,7928000000
TSLA,2024,1,2024-03-31,0,21301000000,1171000000,1390000000
TSLA,2024,2,2024-06-30,0,25500000000,1605000000,1400000000
TSLA,2024,3,2024-09-30,0,25182000000,2717000000,2173000000
TSLA,2024,4,2024-12-31,1,25707000000,1583000000,2128000000
TSLA,2025,1,2025-03-31,0,19335000000,399000000,409000000
TSLA,2025,2,2025-06-30,0,22496000000,923000000,1172000000
TSLA,2025,3,2025-09-30,0,28095000000,1624000000,1373000000
TSLA,2025,4,2025-12-31,1,24901000000,1409000000,840000000
TSLA,2026,1,2026-03-31,0,22387000000,941000000,477000000
TSLA,2026,2,2026-06-30,0,28236000000,398000000,1114000000
TSLA,2026,3,2026-09-30,0,28046000000,718000000,787000000
PLTR,2023,1,2023-03-31,0,525186000,4115000,16802000
PLTR,2023,2,2023-06-30,0,533317000,10074000,28127000
PLTR,2023,3,2023-09-30,0,558159000,39983000,71505000
PLTR,2023,4,2023-12-31,1,608350000,65794000,93391000
PLTR,2024,1,2024-03-31,0,634338000,80881000,105530000
PLTR,2024,2,2024-06-30,0,678134000,105339000,134126000
PLTR,2024,3,2024-09-30,0,725516000,113140000,143525000
PLTR,2024,4,2024-12-31,1,827519000,11043000,79009000
PLTR,2025,1,2025-03-31,0,883855000,176048000,214031000
PLTR,2025,2,2025-06-30,0,1003697000,269317000,326727000
PLTR,2025,3,2025-09-30,0,1181092000,393256000,475599000
PLTR,2025,4,2025-12-31,1,1406802000,575394000,608676000
PLTR,2026,1,2026-03-31,0,1632583000,753998000,870527000
PLTR,2026,2,2026-06-30,0,1935464000,912004000,1061890000
PLTR,2026,3,2026-09-30,0,2313411000,1084845000,1128012000
'''

In [11]:
def read_csv(text: str) -> list[dict]:
    """Parse one of the CSV blocks above into rows, with empty fields as None."""
    rows = list(csv.DictReader(io.StringIO(text.strip())))
    return [{k: (v if v != '' else None) for k, v in row.items()} for row in rows]


def load_table(table: str, rows: list[dict]) -> None:
    """Replace a table's contents with these rows, always in llm_webinar."""
    qualified = f'llm_webinar.{table}'   # named explicitly, so another database is never emptied
    columns = list(rows[0])
    insert = (
        f'INSERT INTO {qualified} (' + ', '.join(columns) + ') VALUES ('
        + ', '.join(f'%({c})s' for c in columns) + ')'
    )

    with s2_conn.cursor() as cur:
        cur.execute(f'TRUNCATE TABLE {qualified}')
        cur.executemany(insert, rows)

    print(f'{table}: {len(rows)} rows')


load_table('companyInfo', read_csv(COMPANIES_CSV))
load_table('financials', read_csv(FINANCIALS_CSV))

In [12]:
%%sql
SELECT ticker, name, industry, exchange FROM companyInfo

### The latest quarter for each company

The same rows the agent will query in step 4, so you know the right answers before you ask. Expect three
rows; revenue is in billions of dollars, rounded.

In [13]:
%%sql
SELECT ticker, fiscal_year, fiscal_quarter,
       ROUND(revenue / 1e9, 2) AS revenue_billions,
       ROUND(net_income / 1e9, 2) AS net_income_billions
FROM financials
WHERE period_end = (SELECT MAX(period_end) FROM financials)
ORDER BY revenue DESC

<a name="connect_s2"></a>
- [Back to Contents](#contents)
## Step 4: Ask questions in English

This is where the notebook stops being a data pipeline. `SQLDatabaseToolkit` gives the model four tools
— list the tables, read a schema, check a query, run a query — and `create_sql_agent` wires them into a
loop: the model writes MySQL, runs it against SingleStore, reads the rows back, and answers in prose.
Expect a short confirmation and a deprecation warning from `langchain-community`; the agent runs in the
cells further down.

**A quirk you may hit.** Reading a `VECTOR` column through SQLAlchemy raises `AttributeError: 'VECTOR'
object has no attribute '_str_impl'` on some builds of the SingleStore dialect, and `SQLDatabase` reads
sample rows from every table. The cell patches that and describes the `embeddings` table by hand, which
also keeps two 1536-dimension vectors out of the prompt.

In [14]:
SQL_AGENT_PREFIX = """You are an agent designed to interact with a SQL database called SingleStore.
This sometimes has Shard and Sort keys in the table schemas, which you can ignore.

Given an input question, create a syntactically correct {dialect} query to run, then look at the
results of the query and return the answer.

If you are asked about similarity questions, you should use the DOT_PRODUCT function.

Here are a few examples of how to use the DOT_PRODUCT function:

Example 1:
Q: how similar are the questions and answers?
A: select question, answer, dot_product(question_embedding, answer_embedding) as similarity from embeddings;

Example 2:
Q: What are the most similar questions in the embeddings table, not including itself?
A: SELECT q1.question as question1, q2.question as question2,
          DOT_PRODUCT(q1.question_embedding, q2.question_embedding) :> float as score
   FROM embeddings q1, embeddings q2
   WHERE q1.question != q2.question
   ORDER BY score DESC LIMIT 5;

Example 3:
Q: In the embeddings table, which rows are from the chatbot?
A: SELECT category, question, answer FROM embeddings WHERE category = 'chatbot';

If you are asked to describe the database, you should run the query SHOW TABLES.

Unless the user specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results. You can order the results by a relevant column to return the
most interesting examples in the database.

Never SELECT the question_embedding or answer_embedding columns, and never SELECT * from the
embeddings table: those columns hold 1536-dimension vectors. Compare them with
DOT_PRODUCT(a, b), which returns a single float, and select that instead. Never query for all
the columns from any table, only the columns relevant to the question.

You have access to tools for interacting with the database. Only use the below tools, and only
use the information returned by them to construct your final answer. You MUST double check your
query before executing it. If you get an error while executing a query, rewrite the query and
try again up to 3 times.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

If the question does not seem related to the database, just return "I don't know" as the answer.
"""

# Older sqlalchemy-singlestoredb builds ship a VECTOR.result_processor that reads
# self._str_impl — an attribute only SQLAlchemy's JSON type defines, while VECTOR subclasses
# BLOB. Reading a VECTOR column through SQLAlchemy therefore raises AttributeError. Current
# builds set it in __init__; this fills it in when they have not, and is a no-op when they have.
try:
    from sqlalchemy import String as _SAString
    from sqlalchemy_singlestoredb.dtypes import VECTOR as _SAVector

    if not hasattr(_SAVector, '_str_impl'):
        _SAVector._str_impl = _SAString()
        print('applied the VECTOR._str_impl compatibility shim')
except ImportError:
    pass   # not using the SingleStore SQLAlchemy dialect

# Describing `embeddings` by hand stops LangChain running SELECT * on it to collect sample
# rows, which is what actually triggered the bug above, and keeps two 1536-dimension vectors
# out of the prompt. The three data tables still get real samples, which is the part that
# helps the agent write good SQL.
EMBEDDINGS_TABLE_INFO = """
CREATE TABLE embeddings (
    id BIGINT,                            -- auto increment
    category VARCHAR(255),                -- 'chatbot' for rows this notebook writes
    question LONGTEXT,
    question_embedding VECTOR(1536, F32), -- never SELECT this column; compare it instead
    answer LONGTEXT,
    answer_embedding VECTOR(1536, F32),   -- never SELECT this column; compare it instead
    created_at DATETIME
)
/* Semantic cache. Compare two vectors with DOT_PRODUCT(a, b), which returns a float. */
"""

db = SQLDatabase.from_uri(
    current_connection_url(),   # read now, not the possibly-expired startup copy
    include_tables=['embeddings', 'companyInfo', 'financials'],
    sample_rows_in_table_info=2,
    custom_table_info={'embeddings': EMBEDDINGS_TABLE_INFO},
)

# Reuse the http_client from step 3 so the agent's calls also skip brotli encoding.
# CHAT_SETTINGS comes from "Choose the models" in step 2. The agent's SQL tools go over the
# Responses API, the only endpoint on which some current models (gpt-6.1-terra among them) call
# tools. The answer then arrives as content blocks, which as_text() below flattens.
llm = ChatOpenAI(model=CHAT_MODEL, http_client=http_client, use_responses_api=True, **CHAT_SETTINGS)

agent_executor = create_sql_agent(
    llm=llm,
    toolkit=SQLDatabaseToolkit(db=db, llm=llm),
    agent_type='tool-calling',
    prefix=SQL_AGENT_PREFIX,
    top_k=3,
    max_iterations=5,
    verbose=True,
)


def as_text(output) -> str:
    """Flatten an agent answer to one string.

    On the Responses API, which langchain-openai uses for GPT-6 models, the answer comes back
    as a list of content blocks such as [{'type': 'text', 'text': '...'}] rather than a plain
    string. The embeddings endpoint rejects a list, and the cache should store readable text.
    """
    if isinstance(output, str):
        return output
    if isinstance(output, list):
        parts = []
        for block in output:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get('type') == 'text':
                parts.append(block.get('text', ''))
        return ''.join(parts)
    return str(output)


def run_agent(question: str) -> str:
    """Ask the SQL agent a question and return just its answer text."""
    return as_text(agent_executor.invoke({'input': question})['output'])


print(f'agent ready over tables: {db.get_usable_table_names()}')

### Put a semantic cache in front of the agent

This is the idea the demo is built around. An agent run costs several seconds and a few thousand tokens,
and many questions users ask are questions somebody already asked, worded differently. So before paying
for a run: embed the question, look for a stored one pointing in nearly the same direction, and if it
scores above a threshold, return its stored answer. Because the vectors live in the same database as the
data, that lookup is a single `DOT_PRODUCT` query over a `VECTOR` column.

`process_user_question()` below is the whole mechanism, and it prints which path it took.

**The similarity threshold is a property of the embedding model.** `0.85` is right for
`text-embedding-3-small`; a number borrowed from a different model can mean the cache silently never
hits and you pay for every answer. The next cell measures it rather than trusting it.

In [15]:
table_name = 'llm_webinar.embeddings'   # named explicitly, like every write in this notebook
# Recalibrated for text-embedding-3-small. 0.97 was carried over from text-embedding-ada-002,
# whose cosine scores sit in a narrow band near the top of the range — unrelated text scored
# ~0.7 there, so 0.97 was a reasonable "nearly the same". The 3-small model spreads scores much
# wider, so the same number now means "almost character-for-character identical" and the cache
# never hits. The calibration cell below measures where paraphrases actually land.
similarity_threshold = 0.85


def to_hex(vector: list[float]) -> str:
    """Pack an embedding as the little-endian float32 bytes a VECTOR(1536, F32) expects."""
    return np.array(vector, dtype='<f4').tobytes().hex()


def process_user_question(question: str, category: str = 'chatbot') -> str:
    print(f'\nQuestion asked: {question}')

    start = time.time()
    question_embedding = get_embeddings([question])[0]
    print(f'  embedding the question: {(time.time() - start) * 1000:.0f} ms')

    lookup = f"""
        SELECT question, answer,
               DOT_PRODUCT(question_embedding, UNHEX(%(qhex)s)) :> float AS score
        FROM {table_name}
        WHERE category = %(category)s
        ORDER BY score DESC
        LIMIT 1
    """

    with s2_conn.cursor() as cur:
        start = time.time()
        cur.execute(lookup, {'qhex': to_hex(question_embedding), 'category': category})
        row = cur.fetchone()
        print(f'  cache lookup: {(time.time() - start) * 1000:.0f} ms')

    if row is not None:
        cached_question, cached_answer, score = row
        print(f'  closest stored question: "{cached_question}" (score {score:.4f})')

        if score > similarity_threshold:
            print('  -> cache hit, returning the stored answer')
            return cached_answer
    else:
        print('  cache is empty')

    print('  -> cache miss, running the agent')
    start = time.time()
    answer = run_agent(question)
    print(f'  agent run: {(time.time() - start) * 1000:.0f} ms')

    if not answer.strip():
        # Nothing to embed, and an empty answer is not worth caching.
        print('  -> the agent returned no answer text, so nothing was cached')
        return 'The agent did not produce an answer. Try rewording the question.'

    answer_embedding = get_embeddings([answer])[0]

    insert = f"""
        INSERT INTO {table_name}
            (category, question, question_embedding, answer, answer_embedding, created_at)
        VALUES
            (%(category)s, %(question)s, UNHEX(%(qhex)s), %(answer)s, UNHEX(%(ahex)s),
             %(created_at)s)
    """

    with s2_conn.cursor() as cur:
        cur.execute(insert, {
            'category': category,
            'question': question,
            'qhex': to_hex(question_embedding),
            'answer': answer,
            'ahex': to_hex(answer_embedding),
            'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        })

    return answer

### Measure the threshold instead of trusting it

A semantic cache has one tuning knob and two ways to get it wrong. Too high and it never hits. Too low
and it returns a stored answer to a question that merely *sounds* similar — which is worse, because a
wrong answer arrives looking exactly like a right one.

So this cell scores a few pairs whose relationship you already know: an identical question, two
rewordings, an unrelated question, and one pair that matters more than the rest. It costs a single
embeddings request. Read the gap between the paraphrases and the unrelated pair — your threshold
belongs inside it. With `text-embedding-3-small` the rewordings land close to `0.9` and the unrelated
pair well below, which is why `0.85` sits where it does. Trust the numbers the cell prints over any
number written in prose, including these.

The first line of output is a check rather than a result: `DOT_PRODUCT` equals cosine similarity only
for unit-length vectors, so those norms should print as `1.0`.

**Watch the last row.** *Bullish* and *bearish* score close together, because an embedding captures what
a sentence is **about** more strongly than which way round it points — the honest limit of caching on
similarity alone.

In [16]:
# Which similarity a paraphrase actually scores is a property of the embedding model, so this
# measures it rather than trusting a threshold inherited from a different one. Costs one
# embeddings request.
CALIBRATION_PAIRS = [
    ('identical but for a stopword', 'describe the database', 'describe database'),
    ('same question, reworded',      'describe the database', 'what tables are in here?'),
    ('same intent, longer',          "what is TSLA's average closing price?",
                                     'give me the mean close for TSLA'),
    ('unrelated',                    'describe the database',
                                     'which tickers have the most bullish news?'),
    ('OPPOSITE meaning',             'which tickers have the most bullish news?',
                                     'which tickers have the most bearish news?'),
]

texts = sorted({text for _, left, right in CALIBRATION_PAIRS for text in (left, right)})
vectors = dict(zip(texts, get_embeddings(texts)))

# DOT_PRODUCT equals cosine similarity only for unit-length vectors. OpenAI's embeddings are
# normalised, which is what lets the SQL lookup use the cheaper DOT_PRODUCT — worth confirming
# rather than assuming, since the scores below would mean something else if it were not true.
norms = [float(np.linalg.norm(v)) for v in vectors.values()]
print(f'vector norms: {min(norms):.4f} to {max(norms):.4f} '
      f'(1.0 means DOT_PRODUCT is cosine similarity)\n')

print(f'{"relationship":<30} score   hit at {similarity_threshold}?')

for label, left, right in CALIBRATION_PAIRS:
    score = float(np.dot(vectors[left], vectors[right]))
    print(f'{label:<30} {score:.4f}  {"YES" if score > similarity_threshold else "no"}')

print(f'\nsimilarity_threshold is {similarity_threshold}. It wants to sit above the unrelated '
      f'pair and below the rewordings;\nif your numbers differ from the commentary above, trust '
      f'these and adjust it.')

### See the cache work

Two questions that mean the same thing, worded differently. The first is a miss: it prints the agent's
whole chain of thought — `Entering new SQL Agent Executor chain`, the queries it writes, the rows it
gets back — and takes around 8 seconds. The second scores above the threshold and comes straight back
from SingleStore in under a second, nearly all of which is the round trip to embed the question. **The
absence of that agent chain on the second question is the entire point:** that is the latency, the
tokens and the money saved.

The cell clears the cache first so the demonstration still works on a re-run — otherwise both
questions hit and there is nothing to compare. The `SELECT` after it shows what got stored.

In [17]:
question_1 = 'describe the database'
question_2 = 'describe database'

# Start from an empty cache, or the demo stops demonstrating anything: on a second run the
# stored copy of question_2 would match itself at 1.0 and "hit" for the wrong reason. Only the
# chatbot rows go — anything else in the table is left alone.
with s2_conn.cursor() as cur:
    cur.execute("DELETE FROM llm_webinar.embeddings WHERE category = 'chatbot'")

answer = process_user_question(question_1)
print(f'\nThe answer is: {answer}')

In [18]:
%%sql
SELECT id, category, question, LEFT(answer, 120) AS answer_start FROM embeddings

Now the second question — the
same request worded differently. Expect a cache hit with its score, no agent chain, and a total
well under a second.

In [19]:
answer = process_user_question(question_2)
print(f'\nThe answer is: {answer}')

<a name="speech"></a>
- [Back to Contents](#contents)
## Step 5: Speak and hear

The agent's answer is text. This step turns it into speech, plays it, and transcribes it back — the
"speak" and "hear" of the title. This cell only defines helpers, so expect no output. **`speak(text)`**
writes an mp3 using the speech model chosen in step 2 and returns its path. **`kernel_value(name)`**
looks up a notebook variable and returns `None` if the cell that sets it has not run, so re-running
step 5 on its own after a kernel restart gives you a message naming the cell to run rather than a bare
`NameError`.

OpenAI is shutting down its separate text-to-speech endpoint on January 6, 2027, so the voice comes
from an audio model over the same chat endpoint as everything else. The model is told to read the text
aloud word for word, and it returns the audio alongside a transcript of what it said.

In [20]:
import base64

from IPython.display import Audio, display


def kernel_value(name: str):
    """A notebook-level value, or None if the cell that sets it has not run.

    Step 5 is the section people re-run on its own — after a kernel restart, or to try a
    different voice — and a restart keeps no variables. Looking the keys up this way lets the
    helpers below say which cell to go back and run, instead of raising a bare NameError from
    somewhere inside themselves.
    """
    return globals().get(name)


def openai_speak(text: str) -> bytes:
    """mp3 bytes from OpenAI's audio model."""
    openai_client = kernel_value('client')

    if openai_client is None:
        raise RuntimeError(
            'No OpenAI client in this kernel. Re-run the "Set your OpenAI key" cell and the cell that '
            'builds `client` in step 2 — a restarted kernel keeps none of its variables.')

    speech_model = kernel_value('SPEECH_MODEL')

    if speech_model is None:
        raise RuntimeError(
            'No OpenAI speech model is available to this key (see "Choose the models" in step 2). '
            'Check https://developers.openai.com/api/docs/models for a current gpt-audio model and '
            'add it to PREFERRED_SPEECH_MODELS.')

    # OpenAI's audio models speak through Chat Completions: ask for audio output, and tell the
    # model to read the text rather than reply to it.
    response = openai_client.chat.completions.create(
        model=speech_model,
        modalities=['text', 'audio'],
        audio={'voice': 'alloy', 'format': 'mp3'},
        messages=[
            {'role': 'system', 'content': (
                "You are a text-to-speech voice. Read the user's message aloud exactly as written, "
                'word for word. Do not answer it, comment on it, or add anything.')},
            {'role': 'user', 'content': text},
        ],
    )

    audio = response.choices[0].message.audio

    if audio is None or not audio.data:
        raise RuntimeError(
            f'{speech_model} returned no audio for this text, so no file was written. Re-run this '
            'cell to try again; if it keeps happening, check the model in "Choose the models".')

    return base64.b64decode(audio.data)


def speak(text: str, path: str = 'output.mp3') -> str:
    """Synthesize `text` to an mp3 at `path` and return the path."""
    audio_bytes = openai_speak(text)

    with open(path, 'wb') as f:
        f.write(audio_bytes)

    print(f"OpenAI {kernel_value('SPEECH_MODEL')}, voice alloy, wrote {len(audio_bytes):,} bytes to {path}")
    return path

Now speak the answer the agent produced in step 4, and play it back. Expect a line naming the
engine and voice, the number of bytes written, and an audio player that starts on its own. The
file is `output.mp3` in the notebook's working directory.

In [21]:
audio_path = speak(answer)
display(Audio(filename=audio_path, autoplay=True))

### Transcribe it back to text

Reading the audio back with the transcription model chosen in step 2 is the "hear" half. Expect a transcript that closely matches
the spoken answer — punctuation and capitalisation will differ, and numbers may come back as words,
which is a fair illustration of what transcription does and does not preserve. In a real application
the input would be a user speaking; round-tripping here means the notebook has an audio file to work
with without asking you for a microphone.

In [22]:
if kernel_value('TRANSCRIBE_MODEL') is None:
    raise RuntimeError(
        'No OpenAI transcription model is available to this key (see "Choose the models" in step 2). '
        'Check https://developers.openai.com/api/docs/models for a current transcription model and '
        'add it to PREFERRED_TRANSCRIBE_MODELS.')

with open(audio_path, 'rb') as audio_file:
    transcript = client.audio.transcriptions.create(model=TRANSCRIBE_MODEL, file=audio_file)

print(transcript.text)

<a name="image"></a>
- [Back to Contents](#contents)
## Step 6: See

The last modality. This step draws a chart straight from the `financials` rows and hands the picture
to a vision model to describe — the way you would write alt text for a chart in a report. Because the
chart is made from rows you can query, you can check the model's description against the numbers:
the image and the data describing it come from the same table. Three cells: ask the agent a question
about growth, draw the chart, describe it. Then one more to speak the description using step 5's
`speak()`.

No new dependency and no new account: the vision model is the same OpenAI client from step 2, and
describing an image is one extra content part in an ordinary chat request.

### Ask the agent about growth

A third question for the agent: which company grew revenue fastest over the past year. Expect an agent
run — a cache miss, since the question is new — and an answer with three percentages. The `%%sql` cell
after it computes the same growth figures directly, so you can check the agent's arithmetic.

In [23]:
question_3 = (
    'Compare each company\'s revenue in its latest quarter with the same quarter a year earlier. '
    'Which company grew fastest, and by what percentage did each of them grow?'
)

answer = process_user_question(question_3)
print(f'\nThe answer is: {answer}')

In [24]:
%%sql
SELECT latest.ticker,
       latest.fiscal_year, latest.fiscal_quarter,
       ROUND(100 * (latest.revenue - prior.revenue) / prior.revenue, 1) AS revenue_growth_pct
FROM financials latest
JOIN financials prior
  ON prior.ticker = latest.ticker
 AND prior.fiscal_quarter = latest.fiscal_quarter
 AND prior.fiscal_year = latest.fiscal_year - 1
WHERE latest.period_end = (SELECT MAX(period_end) FROM financials)
ORDER BY revenue_growth_pct DESC

### Draw the chart

Expect a line chart: one line per company, revenue each quarter as a multiple of where it stood in
Q1 2023. Indexing them is what makes them comparable — Amazon's revenue is about a hundred times
Palantir's, so on a plain dollar axis Palantir would be a flat line along the bottom.

The chart is rendered to PNG bytes in memory and shown from those bytes, the same bytes the next cell
sends to the vision model. Nothing is fetched from the internet.

In [25]:
import base64

from IPython.display import HTML, display

try:
    import matplotlib

    matplotlib.use('Agg')   # render to memory, not to a window
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise RuntimeError('matplotlib is not installed in this kernel. Re-run the %pip install cell '
                       'at the top of the notebook, then re-run this one.') from exc

with s2_conn.cursor() as cur:
    cur.execute("""
        SELECT f.ticker, c.name, f.period_end, f.revenue
        FROM financials f JOIN companyInfo c ON c.ticker = f.ticker
        ORDER BY f.ticker, f.period_end
    """)
    chart_rows = cur.fetchall()

assert chart_rows, 'financials is empty. Run the load cell in step 3 first.'

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)

for ticker in sorted({row[0] for row in chart_rows}):
    series = [row for row in chart_rows if row[0] == ticker]
    base = series[0][3]   # first quarter's revenue, so every line starts at 1.0
    ax.plot([row[2] for row in series], [row[3] / base for row in series],
            marker='o', markersize=3, label=f'{ticker} ({series[0][1]})')

ax.set_title('Quarterly revenue, as a multiple of Q1 2023')
ax.set_ylabel('Revenue relative to Q1 2023')
ax.axhline(1.0, color='grey', linewidth=0.8, linestyle='--')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
fig.tight_layout()

buffer = io.BytesIO()
fig.savefig(buffer, format='png')
plt.close(fig)

chart_png = buffer.getvalue()
chart_data_url = 'data:image/png;base64,' + base64.b64encode(chart_png).decode('ascii')

display(HTML(f'<img src="{chart_data_url}" alt="Quarterly revenue chart" style="max-width: 100%">'))
print(f'{len(chart_png):,} bytes of PNG')

### Describe the chart

Expect a few sentences describing the chart, in the style of alt text, after a few seconds. Compare it
with the chart above and the growth figures from the `%%sql` cell: a good description names which line
climbs fastest and roughly how far, and should agree with the rows.

The image goes to the model in the same message as the instruction, as the base64 `data:` URL the
previous cell built. That is the general pattern for images a kernel holds — there is no public URL for
OpenAI to fetch, and none is needed.

In [26]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{
        'role': 'user',
        'content': [
            {
                'type': 'text',
                'text': (
                    'Describe this chart in three or four sentences, as you would for a screen '
                    'reader: what it measures, and the overall trend of each line, with rough '
                    'values read from the axis. Describe only what the chart shows.'
                ),
            },
            {'type': 'image_url', 'image_url': {'url': chart_data_url}},
        ],
    }],
    # With CHAT_SETTINGS empty the model may reason first, and reasoning counts against this cap.
    max_completion_tokens=250 if CHAT_SETTINGS else 4000,
    **CHAT_SETTINGS,
)

caption = response.choices[0].message.content
print(caption)

Finally, speak the caption with the same `speak()` helper from step 5, writing `caption.mp3` this
time so the answer's audio is still there to compare. All four modalities have now been through one
database: rows in, question in English, answer spoken, image seen and described.

In [27]:
caption_path = speak(caption, path='caption.mp3')
display(Audio(filename=caption_path, autoplay=True))

<a name="troubleshooting"></a>
- [Back to Contents](#contents)
## Troubleshooting

The errors worth knowing about before you need them. The two marked **lies about its cause** are the
reason this table exists.

| What you see | What it means | What to do |
| --- | --- | --- |
| `NameError` or `ModuleNotFoundError` | The cell that defines it has not run in this kernel; a restart keeps no variables | Re-run step 2 from the top |
| `OperationalError: 2628: JWT token expired` | The connection URL's password is a JWT that lasts about an hour | Restart the kernel, which gets a fresh token |
| `no database is selected`, or missing tables | The drop-down at the top points somewhere other than `llm_webinar` | Select the database, re-run step 1, then the connect cell |
| **Lies about its cause:** `APIConnectionError` under a successful `HTTP/1.1 200 OK` | Not the network and not your key — a brotli decoder failed *after* the response arrived | Re-run the OpenAI client cell in step 2, which turns brotli off |
| **Lies about its cause:** the cache never hits, and nothing errors | The similarity threshold does not match the embedding model | Run the calibration cell in step 4 and use the measured gap |
| `AttributeError: 'VECTOR' object has no attribute '_str_impl'` | A SQLAlchemy-only bug in some builds of the SingleStore dialect | Re-run the agent cell in step 4, which patches it in |

<a name="conclusion"></a>
- [Back to Contents](#contents)
## What you built

One database, four modalities. Three years of company financials went into SingleStore; a question
asked in English came back answered; the answer was spoken and heard again as text; and a chart
drawn from those rows was described in a sentence grounded in the row beside it.

None of it needed a second datastore. The cache lookup in step 4 is a `SELECT` with a `DOT_PRODUCT`
against the same tables the agent queries — no export, no sync job, and because the cache is just a
table, you can `SELECT` from it to see why it hit.

<a name="reset"></a>
## Reset the demo

The cell below drops the `llm_webinar` database — tables, rows, cache and all. Step 1 also drops
and recreates tables, but only inside `llm_webinar`; this is the cell that removes everything.

Coming back afterwards costs nothing, in this order: run step 1 to recreate the database and its
tables, select `llm_webinar` in the drop-down, run the connect cell in step 2, then the two load cells
in step 3. To start the *demo* over without touching the data, re-run the cache test in step 4 instead.

In [28]:
%%sql
DROP DATABASE IF EXISTS llm_webinar;

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>